# Teste o Liquid Sense × Gemma 046

Escolha um runtime de CPU no Colab e execute a célula **1. Preparar o Liquid Sense 046**. O download de 2,65 GB fica na máquina da sessão.

Escreva sua pergunta e execute a célula **2. Perguntar ao Liquid Sense 046**. Cada execução abre uma conversa nova. [Modelo avaliado](https://github.com/Felipe9272727/App-pra-ajudar-no-fine-tuning-/releases/tag/liquid-sense-gemma046)

In [ ]:
#@title 1. Preparar o Liquid Sense 046
import hashlib
import json
import os
from pathlib import Path
import platform
import subprocess
import sys
import tempfile
import time
from urllib.request import urlopen
from zipfile import ZipFile

if platform.system() != "Linux" or platform.machine() != "x86_64":
    raise RuntimeError("Use um runtime Linux x86-64.")

repo = "Felipe9272727/App-pra-ajudar-no-fine-tuning-"
release = f"https://github.com/{repo}/releases/download/liquid-sense-gemma046"
revision = "f4b9abd30b3adf72a0f7bae46282e83cd19f6d37"
root = Path(os.environ.get("LIQUID_046_NOTEBOOK_DIR", "/content/liquid-sense-046")).resolve()
root.mkdir(parents=True, exist_ok=True)

def digest(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def fetch(url, path, sha256, limit):
    if path.is_file() and digest(path) == sha256:
        return
    with urlopen(url, timeout=45) as response:
        data = response.read(limit + 1)
    if len(data) > limit or hashlib.sha256(data).hexdigest() != sha256:
        raise RuntimeError(f"Download inválido para {path.name}.")
    path.write_bytes(data)

bundle = root / "LiquidSense-Gemma-046-Linux.zip"
fetch(
    f"{release}/{bundle.name}", bundle,
    "40a7f1c11ee6d906f876419bef8a2d1b09c14e7492bbf2133fd75cded7c70193",
    3_000_000,
)
runtime = root / "runtime"
with ZipFile(bundle) as zip_file:
    zip_file.extractall(runtime)
engine = runtime / "liquid-sense-046"
engine.chmod(0o700)
correction = runtime / "readout.lscal"
correction_sha = "941daca310736af121cdd337317b5e1bddb741b88ea085d684d3760ecc1fecfd"
if digest(correction) != correction_sha:
    raise RuntimeError("Correção 046 inválida.")

parent = root / "LiquidSense-Gemma-045.lsmix"
parent_sha = "e6e4528057543f0bdf35305c87b79197c639f2d77ddfcf140745a62f4441703c"
if not parent.is_file() or digest(parent) != parent_sha:
    recovery = root / "recuperar.py"
    fetch(
        f"https://raw.githubusercontent.com/{repo}/{revision}/fusion046/recover.py",
        recovery,
        "e5cd87bc643cf94905888e6a61f9838fc5b8591832a801a06682b58ed38452a6",
        16_000,
    )
    print("Baixando 2,65 GB de pesos para a máquina da nuvem…", flush=True)
    subprocess.run([sys.executable, str(recovery), str(parent)], check=True)

def executar(pergunta, *, pensar=False, limite=128):
    pergunta = pergunta.strip()
    if not pergunta or len(pergunta) > 12_000:
        raise ValueError("Digite uma pergunta de até 12.000 caracteres.")
    if not 16 <= limite <= 512:
        raise ValueError("Escolha um limite entre 16 e 512 tokens.")
    data = [{
        "id": "colab", "group": "manual", "prompt": pergunta,
        "options": {"thinking": pensar, "reasoning_tokens": 192 if pensar else 0,
                    "response_tokens": limite, "context_tokens": 16384},
    }]

    print("Gerando com o Liquid Sense 046…", flush=True)
    with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", suffix=".json", dir=root) as inputs:
        json.dump(data, inputs, ensure_ascii=False)
        inputs.flush()
        start = time.monotonic()
        process = subprocess.run(
            [str(engine), "calibrated", str(parent), str(correction), correction_sha,
             "bench", inputs.name],
            text=True, capture_output=True, timeout=1200,
        )
        total = time.monotonic() - start
    if process.returncode:
        raise RuntimeError(process.stderr.strip() or "Erro do motor 046.")
    row = json.loads(process.stdout)
    if "error" in row:
        raise RuntimeError(row["error"])
    answer = row["answer"]
    first = answer["first_token_seconds"]
    first_text = f"{first:.2f} s" if first is not None else "indisponível"
    print(f"\nLiquid Sense 046\n{answer['text']}\n")
    print(f"{answer['tokens']} tokens | primeiro token {first_text} | "
          f"inferência {row['seconds']:.2f} s | total com carregamento {total:.2f} s")
    return {
        "resposta": answer["text"], "tokens": answer["tokens"],
        "primeiro_token_s": first, "inferência_s": row["seconds"], "total_s": total,
    }


print("Liquid Sense 046 pronto. Escreva uma pergunta na célula seguinte.")

In [ ]:
#@title 2. Perguntar ao Liquid Sense 046
pergunta = "Qual é o plural de pão? Uma palavra." #@param {type:"string"}
modo_pensar = False #@param {type:"boolean"}
limite_resposta = 128 #@param {type:"integer"}

resultado = executar(pergunta, pensar=modo_pensar, limite=limite_resposta)

O tempo total inclui verificar e carregar os pesos em cada execução. O tempo de inferência separa essa etapa do processamento da pergunta. O motor publicado usa dois threads. A aceleração GPU do notebook exige um motor 046 com backend GPU. As sessões gratuitas têm recursos e duração variáveis. [Limites do Google Colab](https://research.google.com/colaboratory/faq.html)